Goal
- Determine which (subjective) textual characteristics make some deals garner more applications than others
    - Specific focus on mutable characteristics: that which we can change about the text

# Raw deals

In [ ]:
import pandas as pd
import numpy as np
import src.paths

In [2]:
df_deals = pd.read_parquet(
    src.paths.PROCESSED_DATA_DIR / "BARTER_DEALS_MODEL_READY.parquet")

## Reconsolidate product categories

In [ ]:
def clean_tags(tag_set):
    # If the cell is actually just NaN (not a set), return it as is
    if pd.isna(tag_set) and not isinstance(tag_set, (set, list)):
        return np.nan

    # Keep items that are NOT 'UGC' and NOT NaN
    cleaned = {item for item in tag_set if item != 'UGC' and not pd.isna(item)}

    # If the set is now empty (e.g., it was just {'UGC'} or {'UGC', nan}), return NaN
    return cleaned if len(cleaned) > 0 else np.nan


# Apply the cleaning function
df_deals['content_types_clean'] = df_deals['content_types_set'].apply(
    clean_tags)

In [ ]:
import pandas as pd
import numpy as np
import re

# Define the exact word boundaries (\b) so "mother" doesn't trigger on "smother"
# WARNING: The Dutch word "kind" overlaps with the English word "kind" (e.g., "be kind to your skin").
# If you have a lot of English Beauty deals, you might want to replace '\bkind\b' with '\bkindje\b' or '\bkinderen\b' to avoid false positives.
mom_keywords = [
    r'\bbaby\b', r'\bchildren\b', r'\bkids\b', r'\bkind\b', r'\bkinderen\b',
    r'\bparents\b', r'\bvader\b', r'\bmoeder\b', r'\bfather\b', r'\bmother\b',
    r'\bouders\b', r'\btoddler\b', r'\bzwanger\b', r'\bpregnant\b'
]

# Compile the regex pattern for speed
mom_regex = re.compile('|'.join(mom_keywords), re.IGNORECASE)


def assign_macro_category(row):
    tag_set = row['content_types_set']
    deal_type = str(row.get('deal_type', 'unknown')).lower()

    # Combine title and description to search for keywords (adjust column names as needed)
    text_content = str(row.get('deal_text', '')) + " " + \
        str(row.get('description', ''))

    # Handle floats, NaNs, or empty sets
    if not isinstance(tag_set, (set, list)) or pd.isna(tag_set):
        return 'Uncategorized'

    cleaned_tags = {str(t) for t in tag_set if str(
        t).lower() not in ['ugc', 'nan', 'none']}
    if not cleaned_tags:
        return 'Uncategorized'

    # --- 1. The Validated Family & Parenting Category ---
    if 'Mom' in cleaned_tags:
        if cleaned_tags == {'Mom'}:
            return 'Family & Parenting'
        # Check if the text actually contains parenting keywords
        elif mom_regex.search(text_content):
            return 'Family & Parenting'
        else:
            # If it's a fake "Mom" deal, remove the tag so it falls into its true category below
            cleaned_tags.remove('Mom')

    # If a deal didn't have the 'Mom' tag but heavily features the keywords,
    # you can uncomment the next two lines to forcibly categorize it as Parenting:
    # elif mom_regex.search(text_content):
    #     return 'Family & Parenting'

    # If removing 'Mom' left the set empty, it's basically uncategorized
    if not cleaned_tags:
        return 'Other'

    if 'Animals' in cleaned_tags:
        return 'Animals'

    # --- 3. Split Food & Beverage Logic ---
    has_food_drink = any(tag in cleaned_tags for tag in [
                         'Food', 'Drinks', 'Vegan'])
    has_cooking = 'Cooking' in cleaned_tags

    if has_cooking or (has_food_drink and deal_type == 'online'):
        return 'Home Cooking & CPG'
    elif has_food_drink and deal_type == 'physical':
        return 'Dining & Hospitality'
    elif has_food_drink:
        return 'Food & Beverage (Unknown Type)'

    # --- 4. Beauty ---
    if 'Beauty' in cleaned_tags:
        return 'Beauty'

    # --- 5. Fashion ---
    if 'Fashion' in cleaned_tags:
        return 'Fashion'

    # --- 6. Split Entertainment & Experiences Logic ---
    ent_tags = ['Entertainment', 'Experiences', 'Nightlife',
                'Movie', 'Music', 'Travel', 'Activities', 'Photography']
    has_ent = any(tag in cleaned_tags for tag in ent_tags)

    if has_ent and deal_type == 'physical':
        return 'In-Person Experiences'
    elif has_ent and deal_type == 'online':
        return 'Digital Entertainment'
    elif has_ent:
        return 'Entertainment & Experiences (Unknown Type)'

    # --- 7. Lifestyle & Home (The Catch-all) ---
    if any(tag in cleaned_tags for tag in ['Lifestyle', 'Luxury']):
        return 'Lifestyle & Home'

        # --- 2. Tech & Male-Skewed Niche ---
    if any(tag in cleaned_tags for tag in ['Gaming', 'Tech', 'Cars', 'Finance', 'Sport']):
        return 'Tech & Niche'

    return 'Other'


# Apply the new validated function
df_deals['macro_category'] = df_deals.apply(assign_macro_category, axis=1)

## Validation

In [104]:
df_deals['macro_category'].value_counts()

macro_category
Home Cooking & CPG       705
Beauty                   551
Fashion                  409
Digital Entertainment    371
Dining & Hospitality     264
Lifestyle & Home         247
Family & Parenting       172
Animals                  133
In-Person Experiences    127
Tech & Niche              91
Uncategorized             56
Name: count, dtype: int64

In [ ]:
# Convert the sets to sorted strings so identical sets group together properly
df_deals['content_types_str'] = df_deals['content_types_set'].apply(
    lambda x: ', '.join(
        sorted([str(i) for i in x if str(i).lower() not in ['nan', 'none']]))
    if isinstance(x, (set, list)) else 'Uncategorized'
)

# Create the summary suite
validation_suite = df_deals.groupby(['macro_category', 'content_types_str']).agg(
    deal_count=('apps_after_7_days', 'count'),
    avg_apps=('apps_after_7_days', 'mean')
).reset_index()

# Sort to see the largest categories first
validation_suite = validation_suite.sort_values(
    by=['macro_category', 'deal_count'], ascending=[True, False])

# Display the suite (if using Jupyter, just run the variable name)
display(validation_suite)

,macro_category,content_types_str,deal_count,avg_apps
34,Animals,Animals,33,8.242424
1,Animals,"Activities, Animals, Beauty, Cars, Cooking, Dr...",8,4.875000
72,Animals,"Animals, UGC",5,7.800000
33,Animals,"Activities, Animals, Sport",4,1.750000
69,Animals,"Animals, Luxury",3,7.666667
...,...,...,...,...
1121,Tech & Niche,"Gaming, Tech",1,3.000000
1123,Tech & Niche,"Mom, Tech",1,0.000000
1126,Tech & Niche,"Sport, UGC",1,4.000000
1130,Uncategorized,UGC,55,11.581818


In [106]:
validation_suite.to_csv('validation_suite.csv')

Goal
- Determine which (subjective) textual characteristics make some deals garner more applications than others
    - Specific focus on mutable characteristics: that which we can change about the text

In [ ]:
df_deals[df_deals['content_types_set'].apply(lambda x: 'Mom' in x)][[
    'deal_text', 'deal_type', 'content_types_set']]

,deal_text,deal_type,content_types_set
3089,Surprise your dad with Father's Day gifts wort...,online,"{Mom, Lifestyle}"
3093,Je krijgt exclusief het e‑book (4 eiwitrijke w...,online,"{Mom, Vegan, Food, Cooking}"
3105,"Deze speelse, vilten boekjes zijn perfect voor...",online,{Mom}
3109,We bieden Zonnebrandcrème SPF 30 van Food for ...,online,"{Lifestyle, Mom, Beauty, Vegan, Travel, Activi..."
3114,Onze glazen melkopschuimer is een stijlvolle e...,online,"{Lifestyle, Cooking, Mom, UGC, Vegan, Food}"
...,...,...,...
6782,<p>Ontdek&nbsp;het&nbsp;plezier&nbsp;van&nbsp;...,online,"{Mom, Food, Cooking}"
6783,"<p><span>Samen&nbsp;koken,&nbsp;bakken&nbsp;en...",online,"{Mom, Food, Cooking}"
6784,<p><span>Ontdek&nbsp;de&nbsp;krachtige&nbsp;ke...,online,"{Mom, Food, Cooking}"
6797,<p><strong>Geschenke&nbsp;im&nbsp;Wert&nbsp;vo...,online,"{Mom, UGC, Lifestyle}"


# Raw deals

In [ ]:
import pandas as pd
import numpy as np
import src.paths

In [ ]:
df_deals = pd.read_parquet(
    src.paths.PROCESSED_DATA_DIR / "BARTER_DEALS_MODEL_READY.parquet")

## Analyse most/least application deals

- most applications

In [ ]:
df_deals.sort_values('apps_after_7_days', ascending=False)[
    ['deal_title', 'deal_text', 'legacy_partner_id', 'apps_after_7_days']]

,deal_title,deal_text,legacy_partner_id,apps_after_7_days
6803,🇮🇹3-Day Truffle & Wine Experience in Italy €1750,<p><strong>Important:&nbsp;we&nbsp;can&nbsp;pl...,NaN,265
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,NaN,238
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,NaN,183
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,2894.0,182
5265,Unbox Verloren & Retour Postpakketten t.w.v. €...,Een XL doos met verloren & retour postpakkette...,2910.0,177
...,...,...,...,...
3748,"Kom langs bij Woarst, ontvang lunch + winkelte...",Jij komt langs bij Woarst in Arnhem – de lekke...,1875.0,0
3728,Summer Collab: Rechargeable Garden Light,Suspension Positano de New Garden avec source ...,1562.0,0
3701,Mach auch du die Betten sicherer,Du bekommst von mir im Gegenzug Ware im Wert v...,1772.0,0
3113,Huidverstrakking 2.0,Krijg met deze nieuwste trend een strakke mooi...,1593.0,0


- least applications

In [ ]:
df_deals.sort_values('apps_after_7_days', ascending=True)[
    ['deal_title', 'deal_text', 'apps_after_7_days', 'applicants_applications_count']]

,deal_title,deal_text,apps_after_7_days,applicants_applications_count
3113,Huidverstrakking 2.0,Krijg met deze nieuwste trend een strakke mooi...,0,3
3149,Nasal Strip testers,We are looking for German product testers!\nWo...,0,10
5720,Create 3 product review videos of our jewellery,We’re offering 3 pieces of our handcrafted jew...,0,2
5493,MOBVOI TicNote KI Sprachrekorder mit App,Mobvoi TicNote - Der smarte KI-Sprachrekorder ...,0,0
5466,Gourmetten tijdens de feestdagen,Het Kerst Gourmet menu van A-Z Gourmet Service...,0,10
...,...,...,...,...
5265,Unbox Verloren & Retour Postpakketten t.w.v. €...,Een XL doos met verloren & retour postpakkette...,177,493
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,182,436
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,183,192
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,238,416


- NLD only

In [ ]:
mask = df_deals['accepted_countries'].apply(lambda x: 'NLD' in x)
df_deals_nl = df_deals[mask]
df_deals_nl = df_deals_nl[df_deals_nl['deal_type'] == 'online']
a = df_deals_nl.sort_values('apps_after_7_days')[['deal_title', 'deal_text', 'apps_after_7_days',
                                                  'applicants_applications_count']]

a

,deal_title,deal_text,apps_after_7_days,applicants_applications_count
3188,Een unieke nieuwe natuurlijke hondenshampoo,"Een pakket met de Hondenshampoo, de Reinigings...",0,7
4159,Awareness: Schrijven met de Hand | Correctbook,In aanloop naar de Week van Lezen en Schrijven...,0,6
3966,Luxe Tuinverlichting (t.w.v. €200),Je ontvangt één of meerdere armaturen van Ligh...,0,5
5466,Gourmetten tijdens de feestdagen,Het Kerst Gourmet menu van A-Z Gourmet Service...,0,10
5996,Hoe oerpaniek je relatie beïnvloed,Voel je soms dat kleine irritaties plotseling ...,0,0
...,...,...,...,...
5919,Creëer jouw ultieme wellness moment met Rituals,Wij bieden de Rituals The Ritual of Karma Tria...,177,473
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,182,436
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,183,192
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,238,416


# Deals w/ ratings

In [ ]:
# Setup
import yaml
from src.analysis.engine import AnalysisEnvironment

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
env = AnalysisEnvironment(active_analysis='barter_deals')

# Load everything
final_df, og_df = env.load_data()
pm, prompt_map = env.load_prompts(prompt_subfolder='sandbox/BARTER_DEALS')

# REST

In [60]:
# Check the new distribution
print(df_deals['macro_category'].value_counts())

macro_category
Mom                      599
Tech & Niche             552
Home Cooking & CPG       479
Beauty                   431
Fashion                  318
Dining & Hospitality     239
Digital Entertainment    202
Lifestyle & Home         181
In-Person Experiences     69
Uncategorized             56
Name: count, dtype: int64


In [ ]:
df_deals[df_deals['content_types_clean'] == {
    'Lifestyle', 'Tech'}][['deal_text', 'deal_type']]

,deal_text,deal_type
3211,The Calex Smart Indoor IP Camera offers unbeat...,online
3696,We schenken één volledig ‘Jouw Portfolio’-pakk...,online
3791,Wir stellen dir mehrere Lochwände zur Verfügun...,online
4139,Erhalte kostenlos die Tristan Auron Schallzahn...,online
4426,Ontvang de Flystopper 300: een krachtige vlieg...,online
4531,Otiume.nl is een merk gespecialiseerd in stijl...,online
5136,Je ontvangt een Swiftvein Solara wake-up light.,online
5678,"In exchange, you will receive a high-quality h...",online
5834,<p>Ontvang&nbsp;3&nbsp;maanden&nbsp;gratis&nbs...,physical
6405,<p>You&nbsp;receive&nbsp;€300&nbsp;in&nbsp;cre...,online


In [ ]:
df_deals[df_deals['content_types_clean'] ==
         {'Cooking'}][['deal_text', 'deal_type']]

,deal_text,deal_type
3236,"Houd jouw gietijzeren pan, skillet of Dutch ov...",online
3492,We are inviting content creators to visit one ...,online
3531,Bij Downdraft Chef geloven we in een wereld wa...,online
3533,Bij Downdraft Chef geloven we in een wereld wa...,online
3690,We are Pizzabol. High quality and long ferment...,online
3780,Je ontvangt een luxe acacia houten snijplank v...,online
3806,in ruil voor twee UGC-video's ontvang je van o...,online
3821,Jij ontvangt een Koekenpannenset t.w.v. €168 v...,online
5129,"Wir bieten Happy Cheeze - handgefertigte, vega...",online
5403,Je krijgt een luxe acacia houten snijplank t.w...,online


In [42]:
df_deals[df_deals.content_types_clean.isna()]

,applicants_applications_count,content_types,deal_id,main_image,min_social_media_followers,deal_tags,live_since,first_application_at,last_application_at,company_locations,...,log_deal_value,log_online_active_0_7d,log_online_active_7_30d,log_phys_active_0_7d_0_20km,log_phys_active_0_7d_20_50km,log_phys_active_7_30d_0_20km,log_phys_active_7_30d_20_50km,log_global_active_0_7d,log_global_active_7_30d,content_types_clean
3111,28,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019697d1-44ec-00c8-7c86-d799cad44ba2,uploads/deals/019697d0-ac12-ffff-0985-b6cc1090...,1500,None,2025-05-03 20:23:16.585465,2025-05-04 08:21:19.876144,2025-09-19 13:39:51.631259,None,...,5.017280,5.929589,6.626718,0.0,0.0,0.0,0.0,5.924256,6.678342,NaN
3112,47,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019697dd-cc29-00c8-16f0-d20d216883f7,uploads/deals/019697dd-98ac-ffff-4612-efd86a94...,1500,None,2025-05-03 20:36:57.612548,2025-05-04 06:29:24.175618,2025-09-19 13:38:40.049701,None,...,3.713572,5.929589,6.626718,0.0,0.0,0.0,0.0,5.924256,6.678342,NaN
3117,4,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",01969fe6-8873-00c8-ead9-401fd92a3133,uploads/deals/01969fe5-0a4f-ffff-fbd9-b63847ce...,5000,None,2025-05-05 10:03:27.884309,2025-05-05 14:08:57.402484,2025-05-08 18:25:15.218212,None,...,0.000000,5.616771,6.282267,0.0,0.0,0.0,0.0,5.971262,6.654153,NaN
3118,3,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",01969fee-f829-00c8-01a1-e63d1e76ec05,uploads/deals/01969fee-8be1-ffff-40e3-d6cd62b0...,5000,None,2025-05-05 10:12:40.797804,2025-05-05 14:09:12.636788,2025-05-06 12:43:55.695511,None,...,0.000000,5.616771,6.282267,0.0,0.0,0.0,0.0,5.971262,6.654153,NaN
3204,13,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",0196b5d4-e21d-00c8-ccf1-48fed42976b9,uploads/deals/0196b5d3-c91b-ffff-6d94-623709dc...,10000,None,2025-05-09 16:15:50.049109,2025-05-09 17:05:34.975912,2025-05-15 16:41:08.445603,None,...,0.000000,5.153292,5.886104,0.0,0.0,0.0,0.0,5.966147,6.673298,NaN
3205,3,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",0196b5da-8488-00c8-a684-82c19005e799,uploads/deals/0196b5d9-46b2-ffff-18dc-b411e412...,5000,None,2025-05-09 16:21:59.174906,2025-05-09 20:14:42.736404,2025-05-14 12:23:31.489040,None,...,0.000000,5.620401,6.295266,0.0,0.0,0.0,0.0,5.966147,6.673298,NaN
3206,9,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",0196b5df-6fa6-00c8-6bf8-f2bc47963d7f,uploads/deals/0196b5df-4650-ffff-1429-8070ffa9...,5000,None,2025-05-09 16:27:21.538118,2025-05-09 16:42:47.081492,2025-05-14 10:13:49.611355,None,...,0.000000,5.620401,6.295266,0.0,0.0,0.0,0.0,5.966147,6.673298,NaN
3450,149,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",0197124c-1a05-00c8-9be4-a02057ce0939,uploads/deals/0197163e-38d6-ffff-35c5-2c0b357d...,1500,None,2025-05-28 08:09:13.007276,2025-05-28 08:16:28.006209,2026-02-19 09:50:08.794945,None,...,4.795791,6.073045,6.570883,0.0,0.0,0.0,0.0,6.066108,6.595781,NaN
3459,45,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019715f7-4b74-00c8-5312-641373a94b95,uploads/deals/019c294d-d402-ffff-c242-aef3a752...,1500,None,2025-05-28 09:02:00.714332,2025-05-28 09:05:56.025337,2026-03-23 15:22:26.067607,None,...,4.718499,6.066108,6.569481,0.0,0.0,0.0,0.0,6.095825,6.598509,NaN
3461,246,"[{'id': 29, 'name': 'UGC', 'slug': 'Film-Strip'}]",019715fc-cebb-00c8-3e20-deabbb3e3ec0,uploads/deals/01971630-56d8-ffff-c416-2a459687...,1500,None,2025-05-28 09:20:29.777816,2025-05-28 09:56:21.675714,2026-03-21 10:07:15.343069,None,...,4.948760,6.066108,6.569481,0.0,0.0,0.0,0.0,6.095825,6.598509,NaN


In [40]:
df_deals.content_types_clean.value_counts(dropna=False)

content_types_clean
{Beauty}                                                                  181
{Fashion}                                                                 156
{Lifestyle}                                                               120
{Food}                                                                     86
{Mom}                                                                      80
                                                                         ... 
{Mom, Movie, Entertainment}                                                 1
{Lifestyle, Cooking, Luxury, Drinks, Entertainment, Food}                   1
{Lifestyle, Cooking, Luxury, Drinks, Experiences, Entertainment, Food}      1
{Lifestyle, Cooking, Luxury, Entertainment, Food}                           1
{Sport, Cars, Fashion, Activities}                                          1
Name: count, Length: 913, dtype: int64

In [22]:
df_deals[df_deals['content_types_set'] == {'UGC'}]['deal_text']

3111      A box of our best sellers gummies worth of €150
3112          The tastiest and healthiest matcha gummies.
3117    We leveren een bouwlamp op zonen energie in de...
3118    We leveren een foam roller set met als tegenpr...
3204    We leveren een pilates board in de kleur Roze ...
3205    We leveren een jaloezie reiniger in de kleur B...
3206    We leveren een Nekmassage Apparaat Met als teg...
3450    <p>Get&nbsp;50&nbsp;Custom&nbsp;Newspapers&nbs...
3459    <p>Run&nbsp;a&nbsp;custom&nbsp;newspaper&nbsp;...
3461    <p>Create&nbsp;a&nbsp;custom&nbsp;photo&nbsp;b...
3495    We're FireFighterBag.com, creators of bold, fu...
3712    We're looking for UGC creators who'd like to t...
3731    We’re Belter Prints — a rebellious UK-based wa...
3759    Wir bieten Influencern unser neues Schweißnaht...
3761    Wir schicken dir kostenlos unseren Schwanheime...
3812    Wir schicken dir kostenlos unseren beliebten S...
3908    Product - DIY/Home improvement - Owatrol Polyt...
3951    Du bek

In [4]:
df_deals['content_types_set'] = df_deals['content_types_set'].apply(set)

In [15]:
df_deals['content_types_set'].value_counts()

content_types_set
{Beauty}                                           154
{Fashion}                                          151
{Food}                                              79
{Lifestyle}                                         73
{UGC}                                               55
                                                  ... 
{Sport, Gaming, UGC, Entertainment, Activities}      1
{Nightlife, Drinks, Experiences, Entertainment}      1
{Travel, Experiences, Entertainment, Food}           1
{Vegan, Drinks, Experiences, Food}                   1
{Vegan, Lifestyle, Drinks}                           1
Name: count, Length: 1063, dtype: int64

In [14]:
print(df_deals['content_types_set'].count())

3126


In [24]:
a = set(df_deals['content_types_set'].explode())

In [27]:
df_deals['content_types_set'].explode().value_counts()

content_types_set
Lifestyle        1456
UGC              1097
Beauty            773
Fashion           699
Activities        663
Food              645
Luxury            620
Mom               599
Experiences       580
Drinks            454
Entertainment     397
Sport             371
Cooking           350
Vegan             349
Travel            314
Tech              278
Photography       198
Nightlife         169
Animals           136
Gaming            100
Cars               93
Music              93
Finance            83
Movie              45
Name: count, dtype: int64

## Analyse most/least application deals

- most applications

In [22]:
df_deals.sort_values('apps_after_7_days', ascending=False)[
    ['deal_title', 'deal_text', 'legacy_partner_id', 'apps_after_7_days']]

,deal_title,deal_text,legacy_partner_id,apps_after_7_days
6803,🇮🇹3-Day Truffle & Wine Experience in Italy €1750,<p><strong>Important:&nbsp;we&nbsp;can&nbsp;pl...,NaN,265
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,NaN,238
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,NaN,183
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,2894.0,182
5265,Unbox Verloren & Retour Postpakketten t.w.v. €...,Een XL doos met verloren & retour postpakkette...,2910.0,177
...,...,...,...,...
3748,"Kom langs bij Woarst, ontvang lunch + winkelte...",Jij komt langs bij Woarst in Arnhem – de lekke...,1875.0,0
3728,Summer Collab: Rechargeable Garden Light,Suspension Positano de New Garden avec source ...,1562.0,0
3701,Mach auch du die Betten sicherer,Du bekommst von mir im Gegenzug Ware im Wert v...,1772.0,0
3113,Huidverstrakking 2.0,Krijg met deze nieuwste trend een strakke mooi...,1593.0,0


- least applications

In [23]:
df_deals.sort_values('apps_after_7_days', ascending=True)[
    ['deal_title', 'deal_text', 'apps_after_7_days', 'applicants_applications_count']]

,deal_title,deal_text,apps_after_7_days,applicants_applications_count
3113,Huidverstrakking 2.0,Krijg met deze nieuwste trend een strakke mooi...,0,3
3149,Nasal Strip testers,We are looking for German product testers!\nWo...,0,10
5720,Create 3 product review videos of our jewellery,We’re offering 3 pieces of our handcrafted jew...,0,2
5493,MOBVOI TicNote KI Sprachrekorder mit App,Mobvoi TicNote - Der smarte KI-Sprachrekorder ...,0,0
5466,Gourmetten tijdens de feestdagen,Het Kerst Gourmet menu van A-Z Gourmet Service...,0,10
...,...,...,...,...
5265,Unbox Verloren & Retour Postpakketten t.w.v. €...,Een XL doos met verloren & retour postpakkette...,177,493
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,182,436
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,183,192
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,238,416


- NLD only

In [ ]:
mask = df_deals['accepted_countries'].apply(lambda x: 'NLD' in x)
df_deals_nl = df_deals[mask]
df_deals_nl = df_deals_nl[df_deals_nl['deal_type'] == 'online']
a = df_deals_nl.sort_values('apps_after_7_days')[['deal_title', 'deal_text', 'apps_after_7_days',
                                                  'applicants_applications_count']]

a

,deal_title,deal_text,apps_after_7_days,applicants_applications_count
3188,Een unieke nieuwe natuurlijke hondenshampoo,"Een pakket met de Hondenshampoo, de Reinigings...",0,7
4159,Awareness: Schrijven met de Hand | Correctbook,In aanloop naar de Week van Lezen en Schrijven...,0,6
3966,Luxe Tuinverlichting (t.w.v. €200),Je ontvangt één of meerdere armaturen van Ligh...,0,5
5466,Gourmetten tijdens de feestdagen,Het Kerst Gourmet menu van A-Z Gourmet Service...,0,10
5996,Hoe oerpaniek je relatie beïnvloed,Voel je soms dat kleine irritaties plotseling ...,0,0
...,...,...,...,...
5919,Creëer jouw ultieme wellness moment met Rituals,Wij bieden de Rituals The Ritual of Karma Tria...,177,473
5442,Content creator,1. Een vaste vergoeding van €500 per maand (na...,182,436
6585,"Show off our beautiful, comfi and sexy pajamas. 😍",<p>We&nbsp;are&nbsp;offering&nbsp;3&nbsp;sets&...,183,192
6435,Robotstofzuiger SMART met Dweilfunctie en Laad...,<p><strong>De&nbsp;Earkings&nbsp;Robotstofzuig...,238,416


# Deals w/ ratings

In [ ]:
# Setup
import yaml
from src.analysis.engine import AnalysisEnvironment

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [ ]:
env = AnalysisEnvironment(active_analysis='barter_deals')

# Load everything
final_df, og_df = env.load_data()
pm, prompt_map = env.load_prompts(prompt_subfolder='sandbox/BARTER_DEALS')